In [1]:
#%pip install pandas==2.2.3
#%pip install git+https://github.com/pydata/pandas-datareader.git
#%pip install yfinance numpy matplotlib

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib


In [3]:
df = pd.read_csv('../../dataset_completo.csv')
print(df.head())

df.drop(columns=['Date'], inplace=True) #Eliminamos la columna 'Date' ya que no es una característica numérica y no aporta información relevante para el modelo. Además, eliminamos la columna 'target' ya que es la variable que queremos predecir.

         Date  sp500_return    eurusd        vix  petroleo_return  oro_return  \
0  2003-12-31      0.050766  1.259002  18.309999         0.069385    0.047631   
1  2004-01-31      0.017276  1.245206  16.629999         0.016298   -0.032475   
2  2004-02-29      0.012209  1.253007  14.550000         0.031217    0.022960   
3  2004-03-31     -0.016359  1.231300  16.740000         0.049243    0.038561   
4  2004-04-30     -0.016791  1.198294  17.190001         0.045302   -0.094313   

   tipos_fed  inflacion_usa  inflation_eu  desempleo  ...  volatilidad_6m  \
0       0.98       0.020352      0.020207        5.7  ...        0.088742   
1       1.00       0.020263      0.018343        5.7  ...        0.081350   
2       1.01       0.016885      0.016654        5.6  ...        0.079723   
3       1.00       0.017401      0.017175        5.8  ...        0.076144   
4       1.00       0.022926      0.020851        5.6  ...        0.078530   

   momentum_3m  momentum_6m  momentum_12m     lag_

In [4]:
X = df.drop(columns=['target']) #Eliminamos la columna 'Date' ya que no es una característica numérica y no aporta información relevante para el modelo. Además, eliminamos la columna 'target' ya que es la variable que queremos predecir.
Y = df['target']

In [5]:
# Función para generar las particiones preservando las características
# de la serie de tiempo

def division_conjuntos(dataframe, tr_size=0.8, vl_size=0.1, ts_size=0.1 ):
    # Definir número de datos en cada subserie
    N = dataframe.shape[0]
    Ntrain = int(tr_size*N)  # Número de datos de entrenamiento
    Nval = int(vl_size*N)    # Número de datos de validación
    Ntst = N - Ntrain - Nval # Número de datos de prueba

    # Realizar partición
    train = dataframe[0:Ntrain]
    val = dataframe[Ntrain:Ntrain+Nval]
    test = dataframe[Ntrain+Nval:]

    return train, val, test

# Prueba de la función
tr, vl, ts = division_conjuntos(df)

X_train = tr.drop(columns=['target'])
y_train = tr['target']

X_val = vl.drop(columns=['target'])
y_val = vl['target']

X_test = ts.drop(columns=['target'])
y_test = ts['target']

print(f'Entrenamiento: {X_train.shape}')
print(f'Validación: {X_val.shape}')
print(f'Prueba: {X_test.shape}')

Entrenamiento: (212, 26)
Validación: (26, 26)
Prueba: (27, 26)


In [6]:
# ============================================
# RANDOM FOREST - BÚSQUEDA DE HIPERPARÁMETROS
# ============================================

# Definir el espacio de búsqueda
param_grid_rf = {
    'n_estimators': [100, 150, 200, 250, 300],
    'max_depth': [5, 7, 10, 12, 15, 20, None],
    'min_samples_split': [2, 5, 7, 10],
    'min_samples_leaf': [1, 2, 3, 4]
}

# Para series temporales, usar TimeSeriesSplit en lugar de validación cruzada estándar
ts_cv = TimeSeriesSplit(n_splits=3)

# Inicializar el modelo
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

# Búsqueda en rejilla con validación cruzada temporal
grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=ts_cv,
    scoring='neg_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

# Ajustar la búsqueda
grid_search_rf.fit(X_train, y_train)

# Mejores hiperparámetros
print("\n=== RANDOM FOREST ===")
print(f"Mejores hiperparámetros: {grid_search_rf.best_params_}")
print(f"Mejor MSE: {-grid_search_rf.best_score_:.4f}")

Fitting 3 folds for each of 560 candidates, totalling 1680 fits

=== RANDOM FOREST ===
Mejores hiperparámetros: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 250}
Mejor MSE: 0.0080


In [7]:
# ============================================
# RANDOM FOREST - ENTRENAMIENTO FINAL
# ============================================

# Entrenar el modelo final con los mejores hiperparámetros
best_rf = RandomForestRegressor(**grid_search_rf.best_params_, random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)

# Predicciones sobre validación
y_pred_val = best_rf.predict(X_val)

# Métricas de validación
mae_val = mean_absolute_error(y_val, y_pred_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

print(f"\n=== RANDOM FOREST - VALIDACIÓN ===")
print(f"MAE: {mae_val:.4f}")
print(f"RMSE: {rmse_val:.4f}")


=== RANDOM FOREST - VALIDACIÓN ===
MAE: 0.0801
RMSE: 0.0981



=== RANDOM FOREST - VALIDACIÓN ===
MAE: 0.0790
RMSE: 0.0966


In [8]:
joblib.dump(best_rf, 'random_forest_model.pkl')
print("\nModelo Random Forest guardado como 'random_forest_model.pkl'")


Modelo Random Forest guardado como 'random_forest_model.pkl'
